You can download VitalDB_CalBased_Test_Subset.mat from

https://www.kaggle.com/datasets/weinanwangrutgers/pulsedb-balanced-training-and-testing?resource=download&select=VitalDB_Train_Subset.mat

In [4]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
!ls drive/MyDrive/VitalDB_CalBased_Test_Subset.mat

drive/MyDrive/VitalDB_CalBased_Test_Subset.mat


In [12]:
!pip install fastdtw

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567858 sha256=932f2828b588fe51c38ae9a72ba51888f0ba0ee65509db0e598a3bc2903d7c52
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
Successfully built fastdtw


In [35]:
import h5py
import numpy as np
import random
from fastdtw import fastdtw  # We are using this pre-built dtw package because it runs faster than doing it myself.

from collections import defaultdict


def dtw(x, y, dist_fn=lambda a, b: abs(a - b)): # WE ARE NOT USING THIS FUNCTION BECAUSE IT TAKES TOO LONG
    n, m = len(x), len(y)
    D = np.zeros((n+1, m+1)) + np.inf
    D[0, 0] = 0

    # Fill the cost matrix
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = dist_fn(x[i-1], y[j-1])
            D[i, j] = cost + min(D[i-1, j],    # insertion
                                 D[i, j-1],    # deletion
                                 D[i-1, j-1])  # match
    return D[n, m]
def compute_distance_matrix(segments):
    n = len(segments)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            dist,_ = fastdtw(segments[i], segments[j], dist=2)
            D[i, j] = D[j, i] = dist
    return D
def divide_and_conquer_clusters(segments, min_size=2, max_dist=250.0, depth=0):
    n = len(segments)

    # Base case: stop if small or no variance
    if n <= min_size:
        return [segments]

    # Compute pairwise DTW distances
    D = compute_distance_matrix(segments)

    # Find the two most dissimilar segments as cluster seeds
    i, j = np.unravel_index(np.argmax(D), D.shape)
    if D[i, j] < max_dist:
        # Too similar → stop splitting
        return [segments]

    seed_a, seed_b = segments[i], segments[j]
    cluster_a, cluster_b = [seed_a], [seed_b]

    # Assign each remaining segment to nearest seed
    for k in range(n):
        if k == i or k == j:
            continue
        dist_a = dtw(segments[k], seed_a)
        dist_b = dtw(segments[k], seed_b)
        if dist_a < dist_b:
            cluster_a.append(segments[k])
        else:
            cluster_b.append(segments[k])

    # Recursively split each group
    clusters = []
    clusters += divide_and_conquer_clusters(cluster_a, min_size, max_dist, depth+1)
    clusters += divide_and_conquer_clusters(cluster_b, min_size, max_dist, depth+1)

    return clusters

def lb_keogh(s1, s2, r):
    n = len(s1)
    LB_sum = 0
    for i in range(n):
        lower_bound = min(s2[max(0, i - r): min(len(s2), i + r + 1)])
        upper_bound = max(s2[max(0, i - r): min(len(s2), i + r + 1)])
        if s1[i] > upper_bound:
            LB_sum += (s1[i] - upper_bound)**2
        elif s1[i] < lower_bound:
            LB_sum += (s1[i] - lower_bound)**2
    return np.sqrt(LB_sum)
def closest_pair_dtw(series_list, window=5):
    best_dist = float('inf')
    best_pair = None
    n = len(series_list)

    for i in range(n):
        for j in range(i + 1, n):
            s1, s2 = series_list[i], series_list[j]

            # Lower bound pruning
            lb = lb_keogh(s1, s2, window)
            if lb < best_dist:  # only compute DTW if possible improvement
                dist, _ = fastdtw(s1, s2, dist=2)
                if dist < best_dist:
                    best_dist = dist
                    best_pair = (i, j)
    return best_pair, best_dist
def kanadeAlgo(seg):
  maxSum = seg[0]
  maxEnd = seg[0]
  for x in seg:
    maxEnd = max(maxEnd + x, x)
    maxSum = max(maxSum, maxEnd)
  return maxSum


path = 'drive/MyDrive/VitalDB_CalBased_Test_Subset.mat'

with h5py.File(path, 'r') as dataSet:
  subset = dataSet['Subset']
  signals = dataSet['Subset']['Signals'][:1000, 2, :100] # Capping it at 100 for better load times, and because there's limited RAM.
  clusters = divide_and_conquer_clusters(signals)
  for x in clusters:
    pair, dist = closest_pair_dtw(x)
    print(f"Closest pair: {pair}, DTW distance = {dist:.3f}")
  segmentCount = 1
  for y in signals:
    print(f"{segmentCount}: {kanadeAlgo(y):.3f}")
    segmentCount+=1







Closest pair: (0, 1), DTW distance = 71.481
Closest pair: None, DTW distance = inf
Closest pair: (1, 2), DTW distance = 78.235
Closest pair: None, DTW distance = inf
Closest pair: (0, 1), DTW distance = 115.881
Closest pair: None, DTW distance = inf
Closest pair: None, DTW distance = inf
Closest pair: (1, 2), DTW distance = 76.931
Closest pair: None, DTW distance = inf
Closest pair: (0, 1), DTW distance = 842.272
Closest pair: (1, 2), DTW distance = 92.682
Closest pair: (0, 1), DTW distance = 76.180
Closest pair: (1, 2), DTW distance = 83.603
Closest pair: (0, 1), DTW distance = 948.942
Closest pair: (1, 2), DTW distance = 68.768
Closest pair: (1, 2), DTW distance = 83.714
Closest pair: (2, 3), DTW distance = 72.397
Closest pair: (0, 1), DTW distance = 71.173
Closest pair: (0, 1), DTW distance = 70.778
Closest pair: (0, 1), DTW distance = 88.391
Closest pair: (2, 3), DTW distance = 73.950
Closest pair: (0, 1), DTW distance = 73.895
Closest pair: (0, 1), DTW distance = 68.072
Closest pa